# 测试上传查课排班

本脚本用于学风督导队查课排班的安排


## 导入所需库

In [8]:
import re
import pandas
import datetime
from pprint import pprint
from Frame.python3.BaseComponents.DatabaseConnector import DatabaseConnector

## 设置功能函数

利用系统时间获取本周和下一周周一至周日对应的日期

In [9]:
# 获取当日日期，由此判断当日所在周下一周的周一日期
today = datetime.datetime.now().date()
week_monday = today + datetime.timedelta(days=0 - today.weekday())
next_week = today + datetime.timedelta(days=7)
next_week_monday = next_week + datetime.timedelta(days=0 - next_week.weekday())


weekName2Date = {
    "星期一": next_week_monday,
    "星期二": next_week_monday + datetime.timedelta(days=1),
    "星期三": next_week_monday + datetime.timedelta(days=2),
    "星期四": next_week_monday + datetime.timedelta(days=3),
    "星期五": next_week_monday + datetime.timedelta(days=4),
    "Mon": next_week_monday,
    "Tue": next_week_monday + datetime.timedelta(days=1),
    "Wed": next_week_monday + datetime.timedelta(days=2),
    "Thu": next_week_monday + datetime.timedelta(days=3),
    "Fri": next_week_monday + datetime.timedelta(days=4),
}

提供查询和缓存学院名称到学院编号，以及学生姓名到学号的映射

In [10]:
schoolName2ID = dict()
studentName2ID = dict()


def schoolName2IDFunction(database, school_name):
    """
    Use DatabaseConnector to get school_id FROM school_name
    SQL IS "SELECT school_id FROM School WHERE name = school_name"
    """
    if school_name in schoolName2ID:
        return schoolName2ID[school_name]

    DBAffectedRows = database.execute(
        "SELECT school_id FROM School WHERE name = %s", (school_name,)
    )
    if DBAffectedRows == 0:
        raise Exception("School name NOT found")

    schoolName2ID[school_name] = database.fetchall()[0]["school_id"]
    return schoolName2ID[school_name]


def personName2IDFunction(database, student_name):
    """
    Use DatabaseConnector to get student_id FROM student_name
    SQL IS "SELECT student_id FROM MemberBasic WHERE name = student_name"
    """
    if student_name in studentName2ID:
        return studentName2ID[student_name]

    DBAffectedRows = database.execute(
        "SELECT student_id FROM MemberBasic WHERE name = %s", (student_name,)
    )
    if DBAffectedRows != 1:
        raise Exception("Student name NOT only one or NOT found.")

    studentName2ID[student_name] = database.fetchall()[0]["student_id"]
    return studentName2ID[student_name]

### 上传单周或双周课表

In [11]:
def uploadCourseInfo(database):
    # 导入C:\Users\15310\Downloads\风督导队课表单周.xlsx
    df = pandas.read_excel(r"C:\Users\15310\Downloads\学风督导队课表单周.xlsx")
    for i, one_class in enumerate(df.iloc):
        week_period = one_class["上课时间"]
        # 使用re模块匹配 星期[一二三四五]\s*\d-\d
        pattern = re.compile(r"(星期[一二三四五])\s*(\d-\d)")
        pattern_result = pattern.findall(week_period)

        week_name = pattern_result[0][0]
        week = weekName2Date[week_name]
        period = pattern_result[0][1]
        course_name = one_class["课程名称"]

        classroom_name = one_class["上课教室"]
        # 使用re模块匹配 (立人楼|品学楼)\s*[a-cA-C]\d{3}[a-cA-C]?
        pattern = re.compile(r"(立人楼|品学楼)\s*([a-cA-C])(\d{3})-?([a-cA-C]?)")
        pattern_result = pattern.findall(classroom_name)
        building = pattern_result[0][0]
        area = pattern_result[0][1]
        classroom_without_slice = pattern_result[0][2]
        classroom_slice = pattern_result[0][3]

        classroom_id = "1"  # 表示清水河
        if building == "品学楼":
            classroom_id += "1"
        elif building == "立人楼":
            classroom_id += "2"
        if area == "a" or area == "A":
            classroom_id += "1"
        elif area == "b" or area == "B":
            classroom_id += "2"
        elif area == "c" or area == "C":
            classroom_id += "3"
        classroom_id += classroom_without_slice
        if classroom_slice == "a" or classroom_slice == "A":
            classroom_id += "1"
        elif classroom_slice == "b" or classroom_slice == "B":
            classroom_id += "2"
        elif classroom_slice == "c" or classroom_slice == "C":
            classroom_id += "3"
        elif classroom_slice == "":
            classroom_id += "0"

        student_supposed = one_class["上课人数"]
        grade = one_class["年级"]
        school_name = one_class["上课院系"]
        school_id = schoolName2IDFunction(database, school_name)
        order = one_class["编号"]

        DBAffectedRows = database.execute(
            "INSERT INTO Classroom (classroom_id, campus, building, area, room, sit_available) VALUES (%s,%s,%s,%s,%s,0) \
                ON DUPLICATE KEY UPDATE campus = %s, building = %s, area = %s, room = %s;",
            (
                classroom_id,
                "清水河",
                building,
                area,
                classroom_without_slice + classroom_slice,
                "清水河",
                building,
                area,
                classroom_without_slice + classroom_slice,
            ),
            False,
        )
        pprint(
            (
                school_id,
                grade,
                course_name,
                classroom_id,
                student_supposed,
                week.strftime("%Y-%m-%d"),
                period,
                order,
                classroom_name,
            )
        )
        DBAffectedRows = database.execute(
            "INSERT INTO CourseInfo \
                (school_id, grade, name, classroom_id, student_supposed, date, period, course_order, remark) \
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s);",
            (
                school_id,
                str(grade),
                course_name,
                classroom_id,
                student_supposed,
                week.strftime("%Y-%m-%d"),
                period,
                str(order),
                "",
            ),
            False,
        )
        print(
            database.Cursor.mogrify(
                "INSERT INTO CourseInfo \
                (school_id, grade, name, classroom_id, student_supposed, date, period, course_order, remark) \
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s);",
                (
                    school_id,
                    str(grade),
                    course_name,
                    classroom_id,
                    student_supposed,
                    week.strftime("%Y-%m-%d"),
                    period,
                    str(order),
                    "",
                ),
            )
        )

    database.commit()

### 上传查课排班表

In [12]:
def uploadSchedule(database):
    # 导入C:\Users\15310\Downloads\排班.xlsx
    df = pandas.read_excel(
        r"C:\Users\15310\Downloads\排班.xlsx", sheet_name="Schedual Odd", index_col=0
    ).fillna("")
    for week_name in df.columns:
        ds = df[week_name]
        for period_name in ds.index:
            persons = list(map(lambda x: x[:-3], ds[period_name].split("、")))[:-1]
            DBAffectedRows = database.execute(
                "SELECT DISTINCT course_order FROM CourseInfo WHERE date=%s AND period=%s ORDER BY course_order ASC;",
                (weekName2Date[week_name], period_name),
                False,
            )
            if DBAffectedRows < len(persons):
                raise Exception("CourseInfo表中的课程数量不足")
            course_order_list = database.fetchall()
            course_order_list = list(
                map(lambda x: x["course_order"], course_order_list)
            )
            course_person_pairs = zip(course_order_list, persons)
            for CO, PP in course_person_pairs:
                DBAffectedRows = database.execute(
                    f"INSERT INTO CourseCheckSchedule (course_id, schedule_student_id, actual_student_id, remark) \
                    SELECT course_id, '{personName2IDFunction(database,PP)}','{personName2IDFunction(database,PP)}','' \
                    FROM CourseInfo \
                    WHERE date=%s AND period=%s AND course_order=%s;",
                    (weekName2Date[week_name], period_name, CO),
                    False,
                )
                print(
                    database.Cursor.mogrify(
                        f"INSERT INTO CourseCheckSchedule (course_id, schedule_student_id, actual_student_id, remark) \
                    SELECT course_id, '{personName2IDFunction(database,PP)}','{personName2IDFunction(database,PP)}','' \
                        WHERE date=%s AND period=%s AND course_order=%s;",
                        (weekName2Date[week_name], period_name, CO),
                    )
                )
    database.commit()

### 函数主程序

In [13]:
if __name__ == "__main__":
    database = DatabaseConnector()
    database.startCursor()

    uploadCourseInfo(database)
    uploadSchedule(database)

    database.closeCursor()

(4,
 2020,
 '习近平新时代中国特色社会主义思想概论',
 '1111060',
 63,
 '2023-06-12',
 '1-2',
 1,
 '品学楼A106')
INSERT INTO CourseInfo                 (school_id, grade, name, classroom_id, student_supposed, date, period, course_order, remark)             VALUES (4, '2020', '习近平新时代中国特色社会主义思想概论', '1111060', '63', '2023-06-12', '1-2', '1', '');
(4,
 2020,
 '习近平新时代中国特色社会主义思想概论',
 '1111090',
 71,
 '2023-06-12',
 '1-2',
 1,
 '品学楼A109')
INSERT INTO CourseInfo                 (school_id, grade, name, classroom_id, student_supposed, date, period, course_order, remark)             VALUES (4, '2020', '习近平新时代中国特色社会主义思想概论', '1111090', '71', '2023-06-12', '1-2', '1', '');
(4,
 2020,
 '习近平新时代中国特色社会主义思想概论',
 '1111110',
 62,
 '2023-06-12',
 '1-2',
 1,
 '品学楼A111')
INSERT INTO CourseInfo                 (school_id, grade, name, classroom_id, student_supposed, date, period, course_order, remark)             VALUES (4, '2020', '习近平新时代中国特色社会主义思想概论', '1111110', '62', '2023-06-12', '1-2', '1', '');
(12, 2022, '微积分II', '1112040', 

f:\python_venv\.STSA\lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


INSERT INTO CourseCheckSchedule (course_id, schedule_student_id, actual_student_id, remark)                     SELECT course_id, '2020010908023','2020010908023',''                         WHERE date='2023-06-12' AND period='1-2' AND course_order=1;
INSERT INTO CourseCheckSchedule (course_id, schedule_student_id, actual_student_id, remark)                     SELECT course_id, '2020160101031','2020160101031',''                         WHERE date='2023-06-12' AND period='1-2' AND course_order=2;
INSERT INTO CourseCheckSchedule (course_id, schedule_student_id, actual_student_id, remark)                     SELECT course_id, '2021060904021','2021060904021',''                         WHERE date='2023-06-12' AND period='1-2' AND course_order=3;
INSERT INTO CourseCheckSchedule (course_id, schedule_student_id, actual_student_id, remark)                     SELECT course_id, '2020010906026','2020010906026',''                         WHERE date='2023-06-12' AND period='1-2' AND course_order=4;
